# 02 — Feature Engineering & Data Preparation

Ce notebook détaille la transformation des données brutes en un dataset structuré pour le Machine Learning. 

### Étapes clés :
1. **Unification des sources** : Jointure entre les données historiques (Kaggle) et les données live (Scraping API).
2. **Calcul de la Forme (Rolling Stats)** : Création de variables basées sur les 5 derniers matchs.
3. **Calcul de la Puissance (Elo Rating)** : Implémentation d'un système de notation dynamique.
4. **Enrichissement Betting** : Intégration des probabilités implicites des bookmakers.


In [ ]:
import pandas as pd
import numpy as np
import os
import sys
from pathlib import Path

# Ajout du root au path pour importer nos modules si besoin
sys.path.append(str(Path(os.getcwd()).parent.parent))

import psycopg
from psycopg.rows import dict_row

DATABASE_URL = os.getenv("DATABASE_URL", "postgresql://localhost:5432/l1_datalab")


## 1. Chargement des données depuis PostgreSQL

In [ ]:
with psycopg.connect(DATABASE_URL, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT m.id, m.season, m.kickoff, m.home_team_id, m.away_team_id, 
                   m.home_score, m.away_score, m.result,
                   ht.internal_name AS home_team, at_.internal_name AS away_team
            FROM matches m
            JOIN teams ht ON ht.id = m.home_team_id
            JOIN teams at_ ON at_.id = m.away_team_id
            ORDER BY m.kickoff ASC NULLS FIRST
        """)
        df = pd.DataFrame(cur.fetchall())

df['kickoff'] = pd.to_datetime(df['kickoff'], utc=True)
print(f"Total matchs chargés : {len(df)}")
df.head()

## 2. Focus sur l'Elo Rating
L'Elo Rating est notre feature la plus discriminante. Elle permet de quantifier la force d'une équipe non pas sur son nom, mais sur ses résultats réels pondérés par la force de ses adversaires.


In [ ]:
# Visualisation des derniers scores Elo calculés
elos_path = '../features/match_elos.csv'
if os.path.exists(elos_path):
    df_elo = pd.read_csv(elos_path)
    print("Aperçu des scores Elo par match :")
    display(df_elo.head())
else:
    print("⚠️  Le fichier match_elos.csv n'a pas été trouvé. Lancez compute_elo.py d'abord.")

## 3. Assemblage du Dataset ML Final

In [ ]:
# Chargement du dataset généré par notre script create_ml_dataset.py
ml_path = '../features/ml_dataset.csv'
df_ml = pd.read_csv(ml_path)

print(f"Dimensions du dataset final : {df_ml.shape}")
print("\nListe des features générées :")
print(df_ml.columns.tolist())

## 4. Analyse de la cible (Result)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.countplot(x='result', data=df_ml, order=['H', 'D', 'A'], palette='viridis')
plt.title('Distribution des Résultats (H: Domicile, D: Nul, A: Extérieur)')
plt.show()

print("Proportions :")
print(df_ml['result'].value_counts(normalize=True).round(3))